In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv('kaggle_data.csv', index_col=0)
print("Wykryte kolumny:", df.columns.tolist())

Wykryte kolumny: ['statement', 'status']


In [4]:
target_classes = ['Normal', 'Depression', 'Suicidal']
df_filtered = df[df['status'].isin(target_classes)].copy()
df_filtered['label'] = df_filtered['status'].map({
    'Normal': 0, 
    'Depression': 1, 
    'Suicidal': 1
})

In [5]:
df_filtered = df_filtered.dropna(subset=['statement'])
counts = df_filtered['label'].value_counts()
print("Liczba próbek w klasach:\n", counts)

Liczba próbek w klasach:
 label
1    26056
0    16343
Name: count, dtype: int64


In [6]:
min_size = counts.min()
limit = 16000
sample_size = min(min_size, limit)

In [7]:
df_balanced = pd.concat([
    df_filtered[df_filtered['label'] == 0].sample(sample_size, random_state=42),
    df_filtered[df_filtered['label'] == 1].sample(sample_size, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

In [8]:
def clean_statement(text):
    if not isinstance(text, str): # Zabezpieczenie przed wartościami non-string
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#','', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [9]:
df_balanced['cleaned_statement'] = df_balanced['statement'].apply(clean_statement)
df_balanced = df_balanced[df_balanced['cleaned_statement'] != '']
df_balanced[['cleaned_statement', 'label']].to_csv('cleaned_data.csv', index=False)

counts_bal = df_balanced['label'].value_counts()
print("Liczba próbek w klasach:\n", counts_bal)

Liczba próbek w klasach:
 label
1    15998
0    15995
Name: count, dtype: int64
